# DLCZ Feasibility Simulation – Strict Execution Order

This notebook now enforces a linear sequence. You MUST follow these steps on every fresh run or after pressing Reset:

1. Configure: Use the configuration UI to define all required marginals and set CONFIG['tau_pairs'] (list of (varA, varB, tau_target)). Click Apply.
2. Flow 0–3: Builds marginals & Gaussian copula. Fails fast if anything missing or inconsistent.
3. Flow 4–5: Sampling + diagnostics (PIT KS tests; Kendall τ targets). Must pass before physics.
4. Physics Kernel: Define / edit `_compute_metrics_cpu` if not already defined.
5. Flow 6–8: Physics metrics, feasibility assessment, artifact + samples write-out.

Reset Mechanism: Use the Reset cell (next) to clear all readiness flags and internal cached structures. This guarantees you start from a clean state (no residual readiness).

No Silent Fallbacks: Missing marginals or tau_pairs now raise explicit errors. Identity copula is NOT auto-applied. Provide correlations explicitly or adjust CONFIG['tau_pairs'].

Diagnostics Criteria:
- KS p-values must all be ≥ ks_alpha for stochastic marginals.
- Empirical Kendall τ must be within ±tau_tol of targets.

Artifacts Record: schema_hash, seed, tau_pairs, diagnostic thresholds, mc_N, psd_correction_norm, feasibility metrics.

Proceed carefully; each flow cell prints clear status. If a cell prints a Skipped message, resolve prerequisites and re-run in order.


In [ ]:
# Environment / Imports (restored after strict-order refactor)
import numpy as np
import scipy.stats as stats
import json, math, time, os, hashlib
from pathlib import Path
import ipywidgets as W
import warnings

# Basic dependency status
status = {}
for pkg in ['ipywidgets','scipy','plotly']:
    try:
        __import__(pkg)
        status[pkg]='present'
    except ImportError:
        status[pkg]='missing'
print('Dependency status:', status)

# Write requirements (minimal)
reqs = ['numpy','scipy','ipywidgets','plotly']
with open('requirements.txt','w') as fh:
    fh.write('\n'.join(reqs))
print('Wrote requirements.txt')


Dependency status: {'ipywidgets': 'present', 'plotly': 'present', 'scipy': 'present'}
Wrote requirements.txt


In [ ]:
# Reset State (strict sequencing helper)
import ipywidgets as W

def reset_flow_state():
    global WORKING_MARGINALS, FLOW_STATE
    if 'WORKING_MARGINALS' in globals():
        WORKING_MARGINALS = {}
    if 'FLOW_STATE' in globals():
        for k in ['configured','marginals_ready','copula_ready','diagnostics_ok']:
            FLOW_STATE[k] = False
        for k in ['var_order','tau_matrix','chol_L','psd_correction_norm']:
            FLOW_STATE.pop(k, None)
    for name in ['L','Z','Zc','U','U_emp','p_link','p_swap','R','tau_c','metrics','artifact','feas']:
        if name in globals():
            del globals()[name]
    print("[RESET] Cleared state. Sequence required: Configure → Flow 0–3 → Flow 4–5 → Physics Kernel → Flow 6–8.")

_reset_btn = W.Button(description='Reset All State', button_style='warning')
_reset_out = W.Output()

def _on_reset(b):
    with _reset_out:
        _reset_out.clear_output()
        reset_flow_state()

_reset_btn.on_click(_on_reset)
W.VBox([_reset_btn, _reset_out])


In [2]:
# Test config & copula for end-to-end run (safe defaults; overrides CONFIG)
try:
    # Resolve a seed even if the constants cell hasn't run yet
    _seed = int(globals().get('GLOBAL_SEED', 123456789))
    # Minimal CONFIG overlay; SCHEMA provides other defaults
    CONFIG = {
        'preset_id': 'test_preset',
        'alpha_dB_per_km': 0.2,
        'L0_km': 50.0,
        'eta_s': 0.7,
        'eta_w': 0.6,
        'eta_r': 0.6,
        'eta_det': 0.8,
        'V0': 0.9,
        'T2_ms': 150.0,
        'N_modes': 100,
        'n_swaps': 3,
        'kappa_power': 1.0,
        'c_fiber_mps': 2.0e8,
        'connector_loss_dB': 0.5,
        'n_connectors': 10,
        'splice_loss_dB': 0.1,
        'n_splices': 200,
        'filter_loss_dB': 1.0,
        'p_dark': 0.0,
        'raman_rate_per_ns': 0.0,
        'gate_ns': 1.0,
        'geometry': 'central_bsm',
        'kappa_mode': 'power',
        'mc': {'samples': 5000, 'seed': _seed},
        'copula': {'kendall_tau_pairs': [
            ('eta_s','eta_det', 0.25),
            ('eta_s','V0', 0.15),
            ('V0','T2_ms', -0.20)
        ]}
    }
    # Define a clean WORKING_MARGINALS for this test (stochastic subset only)
    # Remove c_fiber_mps from marginals (treat as deterministic) to avoid PIT flake; physics reads it from CONFIG.
    import math as _math
    WORKING_MARGINALS = {
        'eta_s': {'family':'beta','params':{'a':70,'b':30},'unit':'frac'},
        'eta_det': {'family':'beta','params':{'a':75,'b':25},'unit':'frac'},
        'alpha_dB_per_km': {'family':'lognormal','params':{'mu_log':float(_math.log(CONFIG['alpha_dB_per_km'])), 'sigma_log':0.05},'unit':'dB/km'},
        'L0_km': {'family':'uniform','params':{'low':CONFIG['L0_km']*0.8,'high':CONFIG['L0_km']*1.2},'unit':'km'},
        'V0': {'family':'beta','params':{'a':90,'b':10},'unit':'vis'},
        'T2_ms': {'family':'lognormal','params':{'mu_log':float(_math.log(max(CONFIG['T2_ms'],1e-9))),'sigma_log':0.1},'unit':'ms'},
        'N_modes': {'family':'uniform','params':{'low':CONFIG['N_modes']*0.9,'high':CONFIG['N_modes']*1.1},'unit':'count'},
        'n_swaps': {'family':'uniform','params':{'low':max(CONFIG.get('n_swaps',0)-1,0),'high':CONFIG.get('n_swaps',0)+1},'unit':'count'},
        'kappa_power': {'family':'uniform','params':{'low':1.0,'high':1.2},'unit':'power'}
    }
    # Clear prior state for a clean run
    FLOW_STATE = {}
    print('[Test] CONFIG & WORKING_MARGINALS prepared; FLOW_STATE reset.')
except Exception as e:
    print('Test config setup failed:', e)

[Test] CONFIG & WORKING_MARGINALS prepared; FLOW_STATE reset.


In [3]:
# 2) Global Constants, Paths, and Reproducibility Stamp
import numpy as np
from pathlib import Path
import hashlib
import json as _json

GLOBAL_SEED = 123456789
np.random.seed(GLOBAL_SEED)
RNG = np.random.default_rng(GLOBAL_SEED)

ROOT = Path('.')
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
CONFIG_DIR = ROOT / 'config'
CONFIG_DIR.mkdir(exist_ok=True)
(CONFIG_DIR / 'presets').mkdir(parents=True, exist_ok=True)

SCHEMA = {
    'alpha_dB_per_km':    {'type': float, 'unit': 'dB/km', 'min': 0.0, 'default': 0.2},
    'L0_km':              {'type': float, 'unit': 'km',    'min': 0.0, 'default': 25.0},
    'eta_s':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.6},
    'eta_w':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.7},
    'eta_r':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.7},
    'eta_det':            {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.9},
    'V0':                 {'type': float, 'unit': 'vis',   'min': 0.0, 'max': 1.0, 'default': 0.9},
    'T2_ms':              {'type': float, 'unit': 'ms',    'min': 0.01, 'default': 150.0},
    'N_modes':            {'type': float, 'unit': 'count', 'min': 1.0, 'default': 50.0},
    'n_swaps':            {'type': float, 'unit': 'count', 'min': 0.0, 'default': 3.0},
    'kappa_power':        {'type': float, 'unit': 'power', 'min': 0.0, 'default': 1.0},
    'c_fiber_mps':        {'type': float, 'unit': 'm/s',   'min': 1e6, 'default': 2.0e8},
    'geometry':           {'type': str,   'unit': 'enum',  'choices': ['end_detection','central_bsm'], 'default': 'central_bsm'},
    'kappa_mode':         {'type': str,   'unit': 'enum',  'choices': ['power','linear','square'], 'default': 'power'},
    'connector_loss_dB':  {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.5},
    'n_connectors':       {'type': int,   'unit': 'count', 'min': 0,   'default': 2},
    'splice_loss_dB':     {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.1},
    'n_splices':          {'type': int,   'unit': 'count', 'min': 0,   'default': 0},
    'filter_loss_dB':     {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.0},
    'p_dark':             {'type': float, 'unit': 'prob',  'min': 0.0, 'max': 1.0, 'default': 0.0},
    'raman_rate_per_ns':  {'type': float, 'unit': 'prob/ns','min': 0.0, 'default': 0.0},
    'gate_ns':            {'type': float, 'unit': 'ns',    'min': 0.0, 'default': 1.0},
}

def _schema_for_hash():
    def clean(v):
        if isinstance(v, dict):
            return {k: clean(vv) for k, vv in v.items()}
        if isinstance(v, (list, tuple)):
            return [clean(x) for x in v]
        if isinstance(v, type):
            return v.__name__
        return v
    return clean(SCHEMA)

def reproducibility_stamp(extra=None):
    h = hashlib.sha256()
    h.update(str(GLOBAL_SEED).encode())
    serializable_schema = _schema_for_hash()
    h.update(_json.dumps(serializable_schema, sort_keys=True).encode())
    if extra:
        h.update(_json.dumps(extra, sort_keys=True).encode())
    return h.hexdigest()[:16]

print('Reproducibility stamp:', reproducibility_stamp())

Reproducibility stamp: 92ebdda8929a388f


In [4]:
# Flow Step 0–3 (Strict): Bootstrap → Config validation → Marginal binding → Copula assembly
# Strict ordering enforced: requires WORKING_MARGINALS to be defined by configuration cell; no defaults created here.

try:
    import numpy as _np, json as _json, hashlib as _hashlib
    from pathlib import Path as _Path
    from math import pi
    from scipy.stats import beta as _beta, lognorm as _lognorm, uniform as _uniform, truncnorm as _truncnorm
    from scipy.stats import norm as _norm

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    # Bootstrap
    device = str(globals().get('SELECTED_DEVICE', 'cpu') or 'cpu')
    dtype = str(_np.array(0.0).dtype)
    seed = int(globals().get('GLOBAL_SEED', 123456789))
    print(f"[Flow 0] seed={seed} device={device} dtype={dtype}")
    FLOW_STATE.update({'seed': seed, 'device': device, 'dtype': dtype, 'warnings': []})

    # Merge CONFIG over SCHEMA defaults
    raw_cfg = globals().get('CONFIG') or {}
    cfg = {}
    for k, meta in SCHEMA.items():
        cfg[k] = raw_cfg.get(k, meta.get('default'))
    # Nested mc
    mc_cfg = raw_cfg.get('mc', {})
    if 'mc_samples' in raw_cfg and 'samples' not in mc_cfg:
        mc_cfg['samples'] = raw_cfg['mc_samples']
    if 'mc_seed' in raw_cfg and 'seed' not in mc_cfg:
        mc_cfg['seed'] = raw_cfg['mc_seed']
    if 'seed' not in mc_cfg:
        mc_cfg['seed'] = seed
    if 'samples' not in mc_cfg:
        mc_cfg['samples'] = 5000
    cfg['mc'] = mc_cfg

    REQUIRED = ['alpha_dB_per_km','L0_km','eta_s','eta_w','eta_r','eta_det','V0','T2_ms','N_modes','connector_loss_dB','n_connectors','splice_loss_dB','n_splices','filter_loss_dB','c_fiber_mps','geometry','mc']
    messages=[]; table=[]; ok=True
    for key in REQUIRED:
        val = cfg.get(key)
        unit = SCHEMA.get(key,{}).get('unit','')
        lo = SCHEMA.get(key,{}).get('min'); hi = SCHEMA.get(key,{}).get('max')
        if val is None:
            ok=False; messages.append(f'Missing required key: {key}')
        else:
            if key!='mc' and isinstance(val,(int,float)):
                if lo is not None and val < lo: ok=False; messages.append(f'{key}={val} < {lo}')
                if hi is not None and val > hi: ok=False; messages.append(f'{key}={val} > {hi}')
        table.append((key,val,unit,lo,hi))

    # Derive n_swaps if needed
    if cfg.get('total_distance_km') is not None and cfg.get('n_swaps') is None and cfg.get('L0_km'):
        try:
            cfg['n_swaps'] = max(0,int(round(cfg['total_distance_km']/cfg['L0_km']) - 1))
            messages.append(f"Derived n_swaps={cfg['n_swaps']}")
        except Exception:
            ok=False; messages.append('Failed deriving n_swaps')
    elif cfg.get('n_swaps') is None:
        cfg['n_swaps'] = 0

    print('[Flow 1] Schema table (key | value | unit | min..max):')
    for key,val,unit,lo,hi in table:
        print(f'  {key} | {val} | {unit} | [{lo},{hi}]')
    if messages:
        print('[Flow 1] Notes:'); [print(' ',m) for m in messages]

    FLOW_STATE['config_ok']=ok
    hash_basis={k:cfg[k] for k in cfg if k!='mc'}; hash_basis['mc_samples']=cfg['mc']['samples']
    FLOW_STATE['schema_hash']=_hashlib.sha256(_json.dumps(hash_basis,sort_keys=True).encode()).hexdigest()[:16]
    FLOW_STATE['preset_id']=cfg.get('preset_id')

    if not ok:
        print('[Flow 0–3] Aborting after config validation failure.')
    else:
        # Require WORKING_MARGINALS prepared by configuration cell
        if 'WORKING_MARGINALS' not in globals() or not WORKING_MARGINALS:
            FLOW_STATE['marginals_ready']=False
            print('[Flow 2] Skipped: WORKING_MARGINALS not set. Run the configuration cell first.')
        else:
            marg=WORKING_MARGINALS; bound={}
            def _bind_margin(key,spec):
                fam=(spec.get('family') or spec.get('dist') or '').lower(); params=spec.get('params',{}); unit=spec.get('unit','')
                if fam=='beta': a=float(params['a']); b=float(params['b']); cdf=lambda x:_beta.cdf(x,a,b); ppf=lambda u:_beta.ppf(u,a,b)
                elif fam=='lognormal': mu=float(params['mu_log']); sigma=float(params['sigma_log']); s=sigma; scale=_np.exp(mu); cdf=lambda x:_lognorm.cdf(x,s,scale=scale); ppf=lambda u:_lognorm.ppf(u,s,scale=scale)
                elif fam=='uniform': low=float(params['low']); high=float(params['high']); cdf=lambda x:_uniform.cdf(x,loc=low,scale=high-low); ppf=lambda u:_uniform.ppf(u,loc=low,scale=high-low)
                elif fam in ('truncnorm','truncated_normal'): lo=float(params['low']); hi=float(params['high']); loc=float(params.get('loc',0.0)); scale=float(params.get('scale',1.0)); a=(lo-loc)/scale; b=(hi-loc)/scale; cdf=lambda x:_truncnorm.cdf(x,a,b,loc=loc,scale=scale); ppf=lambda u:_truncnorm.ppf(u,a,b,loc=loc,scale=scale)
                else: raise ValueError(f'Unsupported margin family {fam}')
                return {'cdf':cdf,'ppf':ppf,'unit':unit,'family':fam,'params':params}
            try:
                for k,s in marg.items(): bound[k]=_bind_margin(k,s)
                FLOW_STATE['marginals_bound']=bound; FLOW_STATE['marginals_ready']=True
                FLOW_STATE['var_order']=list(bound.keys())
                print('[Flow 2] Marginals bound:');
                for k,b in bound.items(): print(f"  {k}: {b['family']} params={b['params']} unit={b['unit']}")
            except Exception as e:
                FLOW_STATE['marginals_ready']=False; print('[Flow 2] Marginal binding failed:',e)

        # Copula assembly requires tau pairs configured
        tau_pairs = raw_cfg.get('copula',{}).get('kendall_tau_pairs', [])
        FLOW_STATE['tau_pairs']=[(a,b,float(t)) for a,b,t in tau_pairs]
        if FLOW_STATE.get('marginals_ready') and tau_pairs:
            var_order=FLOW_STATE['var_order']
            idx={k:i for i,k in enumerate(var_order)}; tau_mat=_np.zeros((len(var_order),len(var_order)))
            for a,b,t in tau_pairs:
                if a in idx and b in idx: ia,ib=idx[a],idx[b]; tau_mat[ia,ib]=tau_mat[ib,ia]=float(t)
            rho_psd,corr_norm=tau_to_rho_matrix(tau_mat)
            FLOW_STATE['Sigma_psd']=rho_psd; FLOW_STATE['psd_correction_norm']=float(corr_norm)
            try:
                L=robust_cholesky(rho_psd) if 'robust_cholesky' in globals() else _np.linalg.cholesky(rho_psd+1e-12*_np.eye(len(var_order)))
                FLOW_STATE['chol_L']=L; FLOW_STATE['copula_ready']=True
            except Exception as e:
                FLOW_STATE['copula_ready']=False; print('[Flow 3] Cholesky failed:',e)
            eig=_np.linalg.eigvalsh(rho_psd)
            print('[Flow 3] Copula:'); print(f'  pair_count={len(tau_pairs)} avg|τ|={(_np.mean(_np.abs([t for _,_,t in tau_pairs])) if tau_pairs else 0.0):.3f}')
            print(f'  PSD correction norm={float(corr_norm):.3e} eig[min,max]=[{eig.min():.3e},{eig.max():.3e}]')
            print('  Σ_psd[0:3,0:3]='); print(_np.array2string(rho_psd[:3,:3],precision=3,suppress_small=True))
        else:
            FLOW_STATE['copula_ready']=False
            if not FLOW_STATE.get('marginals_ready'):
                print('[Flow 3] Skipped: marginals not ready.')
            elif not tau_pairs:
                print('[Flow 3] Skipped: no copula kendall_tau_pairs configured. Configure copula first.')
except Exception as e:
    print('Flow Step 0–3 error:', e)


[Flow 0] seed=123456789 device=cpu dtype=float64
[Flow 1] Schema table (key | value | unit | min..max):
  alpha_dB_per_km | 0.2 | dB/km | [0.0,None]
  L0_km | 50.0 | km | [0.0,None]
  eta_s | 0.7 | frac | [0.0,1.0]
  eta_w | 0.6 | frac | [0.0,1.0]
  eta_r | 0.6 | frac | [0.0,1.0]
  eta_det | 0.8 | frac | [0.0,1.0]
  V0 | 0.9 | vis | [0.0,1.0]
  T2_ms | 150.0 | ms | [0.01,None]
  N_modes | 100 | count | [1.0,None]
  connector_loss_dB | 0.5 | dB | [0.0,None]
  n_connectors | 10 | count | [0,None]
  splice_loss_dB | 0.1 | dB | [0.0,None]
  n_splices | 200 | count | [0,None]
  filter_loss_dB | 1.0 | dB | [0.0,None]
  c_fiber_mps | 200000000.0 | m/s | [1000000.0,None]
  geometry | central_bsm | enum | [None,None]
  mc | {'samples': 5000, 'seed': 123456789} |  | [None,None]
[Flow 2] Marginals bound:
  eta_s: beta params={'a': 70, 'b': 30} unit=frac
  eta_det: beta params={'a': 75, 'b': 25} unit=frac
  alpha_dB_per_km: lognormal params={'mu_log': -1.6094379124341003, 'sigma_log': 0.05} unit=d

In [19]:
# Flow Step 4–5 (Strict): Joint sampling (Gaussian copula) → Diagnostics
# Requires Flow 0–3 to have set FLOW_STATE['marginals_ready']=True and FLOW_STATE['copula_ready']=True.

try:
    import numpy as _np
    from scipy.stats import norm as _norm
    from scipy.stats import kstest as _kstest
    from scipy.stats import kendalltau as _kendalltau

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    if not FLOW_STATE.get('marginals_ready', False):
        print('[Flow 4] Skipped: marginals not ready. Run configuration and Flow 0–3 first.')
    elif not FLOW_STATE.get('copula_ready', False):
        print('[Flow 4] Skipped: copula not ready. Configure copula kendall_tau_pairs and run Flow 0–3.')
    else:
        # thresholds from CONFIG
        diag_cfg = (globals().get('CONFIG') or {}).get('diag', {}) or {}
        ks_alpha = float(diag_cfg.get('ks_alpha', 0.05))
        tau_tol = float(diag_cfg.get('tau_tol', 0.05))
        min_N_for_tau = int(diag_cfg.get('min_N_for_tau', 5000))
        force_proceed = bool(diag_cfg.get('force_proceed', False))

        N = int(FLOW_STATE.get('mc_N', 0) or (globals().get('CONFIG') or {}).get('mc',{}).get('samples', 0) or 0)
        if N <= 0:
            print('[Flow 4] Skipped: mc sample size not set. Set CONFIG["mc"]["samples"] and rerun Flow 0–3.')
        else:
            seed = int(FLOW_STATE.get('seed', globals().get('GLOBAL_SEED', 123456789)))
            rng = _np.random.default_rng(seed)
            L = FLOW_STATE['chol_L']
            var_order = FLOW_STATE['var_order']
            bound = FLOW_STATE['marginals_bound']

            # Z -> Zc -> U
            Z = rng.standard_normal(size=(N, len(var_order)))
            Zc = Z @ L.T
            U = _norm.cdf(Zc)
            U = _np.clip(U, 1e-12, 1-1e-12)

            # Map to samples via PPF
            X = {}
            for j, key in enumerate(var_order):
                ppf = bound[key]['ppf']
                X[key] = _np.asarray(ppf(U[:, j]))
            FLOW_STATE['samples_X'] = X
            FLOW_STATE['U'] = U
            FLOW_STATE['mc_N'] = N

            # Show head
            print('[Flow 4] Samples head:')
            head_rows = min(5, N)
            for key in var_order:
                unit = bound[key].get('unit','')
                print(f'  {key} [{unit}]:', _np.array2string(X[key][:head_rows], precision=4))

            # Diagnostics
            print('[Flow 5] Diagnostics: PIT and Kendall τ checks')
            pit_results = {}
            ks_fail = []
            for j, key in enumerate(var_order):
                # Skip constants in PIT
                if bound.get(key,{}).get('family') in ('const','constant') or _np.std(X[key]) < 1e-15:
                    continue
                cdf = bound[key]['cdf']
                U_emp = _np.clip(cdf(X[key]), 1e-12, 1-1e-12)
                stat, pval = _kstest(U_emp, 'uniform')
                pit_results[key] = {'ks_stat': float(stat), 'p_value': float(pval)}
                if pval < ks_alpha:
                    ks_fail.append(key)

            tau_pairs = FLOW_STATE.get('tau_pairs', [])
            tau_results = []
            tau_fail = []
            for (i_key, j_key, tau_target) in tau_pairs:
                if i_key in X and j_key in X:
                    tau_emp, _ = _kendalltau(X[i_key], X[j_key])
                    delta = abs(float(tau_emp) - float(tau_target))
                    tau_results.append((i_key, j_key, float(tau_target), float(tau_emp), float(delta)))
                    if delta > tau_tol and N >= min_N_for_tau:
                        tau_fail.append((i_key, j_key))

            print('  PIT results (KS p-values):')
            for k,v in pit_results.items():
                print(f"    {k}: p={v['p_value']:.3f}")
            if tau_results:
                print('  Kendall τ targets vs empirical:')
                for i_key,j_key,tau_t,tau_e,delta in tau_results:
                    print(f'    ({i_key},{j_key}): target={tau_t:+.3f}, emp={tau_e:+.3f}, |Δ|={delta:.3f}')

            diagnostics_ok = (len(ks_fail)==0) and (len(tau_fail)==0)
            if not diagnostics_ok and force_proceed:
                print('[Flow 5] Diagnostics failed, but force_proceed=True; continuing.')
                diagnostics_ok = True
            FLOW_STATE['diagnostics_ok'] = diagnostics_ok
            if not diagnostics_ok:
                print('[Flow 5] Diagnostics failed; do not run physics. Adjust marginals/copula and rerun Flow 0–3, then Flow 4–5.')
            else:
                print('[Flow 5] Diagnostics OK.')
except Exception as e:
    print('Flow Step 4–5 error:', e)


[Flow 4] Samples head:
  eta_s [frac]: [0.6904 0.7082 0.7549 0.7007 0.619 ]
  eta_det [frac]: [0.7693 0.7867 0.7839 0.772  0.7113]
  alpha_dB_per_km [dB/km]: [0.201  0.2092 0.1896 0.2104 0.2242]
  L0_km [km]: [59.0638 47.7902 42.1063 40.4391 47.4709]
  V0 [vis]: [0.9133 0.8703 0.9114 0.9205 0.8653]
  T2_ms [ms]: [164.5906 170.6174 171.2459 148.4089 157.4569]
  N_modes [count]: [109.1943  92.9505  92.7792 100.4404 101.3435]
  n_swaps [count]: [3.0981 3.995  3.26   2.1779 3.9031]
  kappa_power [power]: [1.0733 1.198  1.0287 1.102  1.1923]
[Flow 5] Diagnostics: PIT and Kendall τ checks
  PIT results (KS p-values):
    eta_s: p=0.299
    eta_det: p=0.493
    alpha_dB_per_km: p=0.647
    L0_km: p=0.643
    V0: p=0.646
    T2_ms: p=0.806
    N_modes: p=0.154
    n_swaps: p=0.913
    kappa_power: p=0.096
  Kendall τ targets vs empirical:
    (eta_s,eta_det): target=+0.250, emp=+0.246, |Δ|=0.004
    (eta_s,V0): target=+0.150, emp=+0.156, |Δ|=0.006
    (V0,T2_ms): target=-0.200, emp=-0.199, |Δ|

In [20]:
# Flow Step 6–8 (Strict): Physics → Feasibility → Reporting
# Requires diagnostics_ok from Flow 4–5 and _compute_metrics_cpu defined in physics kernel cell.

try:
    import numpy as _np
    import time as _time
    from pathlib import Path as _Path
    import json as _json
    from math import sqrt
    import hashlib as _hashlib

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    if not FLOW_STATE.get('diagnostics_ok', False):
        print('[Flow 6–8] Skipped: diagnostics not passed. Run Flow 4–5 after successful Flow 0–3.')
    elif '_compute_metrics_cpu' not in globals():
        print('[Flow 6–8] Skipped: physics kernel not defined. Execute the physics kernel cell first.')
    else:
        X = FLOW_STATE.get('samples_X')
        if not isinstance(X, dict):
            print('[Flow 6] Skipped: samples_X missing. Run Flow 4–5.')
        else:
            try:
                metrics = _compute_metrics_cpu(X)
            except Exception as e:
                print('[Flow 6] Physics computation failed:', e)
                raise

            R = _np.asarray(metrics.get('R', []))
            tau_c = _np.asarray(metrics.get('tau_c', [])) if 'tau_c' in metrics else _np.array([])
            p_link = _np.asarray(metrics.get('p_link', [])) if 'p_link' in metrics else _np.array([])
            p_swap = _np.asarray(metrics.get('p_swap', [])) if 'p_swap' in metrics else _np.array([])

            def _quantiles(arr):
                arr = _np.asarray(arr)
                if arr.size == 0:
                    return {"p50": _np.nan, "p10": _np.nan, "p90": _np.nan}
                q10, q50, q90 = _np.percentile(arr, [10,50,90])
                return {"p50": float(q50), "p10": float(q10), "p90": float(q90)}

            print('[Flow 6] Distribution summaries:')
            if p_link.size:
                q = _quantiles(p_link); print(f"  p_link: p50={q['p50']:.3e}, p10={q['p10']:.3e}, p90={q['p90']:.3e}")
            if p_swap.size:
                q = _quantiles(p_swap); print(f"  p_swap: p50={q['p50']:.3e}, p10={q['p10']:.3e}, p90={q['p90']:.3e}")
            if R.size:
                q = _quantiles(R); print(f"  R: p50={q['p50']:.3e}, p10={q['p10']:.3e}, p90={q['p90']:.3e}")
            if tau_c.size:
                q = _quantiles(tau_c); print(f"  tau_c: p50={q['p50']:.3e}, p10={q['p10']:.3e}, p90={q['p90']:.3e}")

            def _wilson_interval(k, n, z=1.96):
                if n == 0: return (0.0,0.0,0.0)
                phat = k/n; denom = 1 + z*z/n
                center = (phat + z*z/(2*n))/denom
                margin = (z * sqrt((phat*(1-phat) + z*z/(4*n))/n))/denom
                return (phat, max(0.0, center - margin), min(1.0, center + margin))

            R_target = FLOW_STATE.get('R_target', 1e-6)
            k = int(_np.sum(R >= R_target)) if R.size else 0
            phat, lo, hi = _wilson_interval(k, int(R.size))
            print('[Flow 7] Feasibility:')
            print(f'  P(R ≥ {R_target:.3g}) = {phat:.3%} (95% CI [{lo:.3%}, {hi:.3%}])')
            feas = {'R_target': float(R_target), 'p_ge_target': float(phat), 'ci95': [float(lo), float(hi)]}

            if tau_c.size and 'T2_ms' in X:
                try:
                    c_factor = FLOW_STATE.get('decoherence_c', 1.0)
                    tau_c_ms = _np.asarray(tau_c) * 1e3
                    survive = (_np.asarray(X['T2_ms']) >= c_factor * tau_c_ms)
                    k2 = int(_np.sum(survive))
                    ph2, lo2, hi2 = _wilson_interval(k2, int(len(survive)))
                    print(f'  P(T2 ≥ {c_factor}·tau_c) = {ph2:.3%} (95% CI [{lo2:.3%}, {hi2:.3%}])')
                    feas['T2_vs_tau_c'] = {'c': float(c_factor), 'p': float(ph2), 'ci95': [float(lo2), float(hi2)]}
                except Exception:
                    pass

            RESULTS_DIR.mkdir(parents=True, exist_ok=True)
            ts = _time.strftime('%Y%m%dT%H%M%S')

            schema_hash = FLOW_STATE.get('schema_hash')
            if not schema_hash:
                try:
                    schema_repr = _json.dumps(SCHEMA, sort_keys=True).encode()
                    schema_hash = _hashlib.sha256(schema_repr).hexdigest()[:16]
                    FLOW_STATE['schema_hash'] = schema_hash
                except Exception:
                    schema_hash = None

            diag_cfg = (globals().get('CONFIG') or {}).get('diag', {}) or {}
            ks_alpha = float(diag_cfg.get('ks_alpha', 0.05))
            tau_tol = float(diag_cfg.get('tau_tol', 0.05))
            min_N_for_tau = int(diag_cfg.get('min_N_for_tau', 5000))

            artifact = {
                'schema_hash': schema_hash,
                'preset_id': FLOW_STATE.get('preset_id'),
                'seed': int(FLOW_STATE.get('seed', -1)),
                'device': FLOW_STATE.get('device', 'cpu'),
                'dtype': FLOW_STATE.get('dtype', 'float64'),
                'Sigma_psd_correction_norm': float(FLOW_STATE.get('psd_correction_norm', 0.0)) if FLOW_STATE.get('psd_correction_norm') is not None else None,
                'timestamp': ts,
                'warnings': FLOW_STATE.get('warnings', []),
                'feasibility': feas,
                'metrics_summary': {
                    'R_quantiles': _quantiles(R) if R.size else None,
                    'p_link_quantiles': _quantiles(p_link) if p_link.size else None,
                    'tau_c_quantiles': _quantiles(tau_c) if tau_c.size else None,
                },
                'provenance': {
                    'tau_pairs': FLOW_STATE.get('tau_pairs', []),
                    'var_order': FLOW_STATE.get('var_order', []),
                    'diag_thresholds': {
                        'ks_alpha': ks_alpha,
                        'tau_tol': tau_tol,
                        'min_N_for_tau': min_N_for_tau
                    },
                    'mc_N': int(FLOW_STATE.get('mc_N', 0)),
                    'psd_correction_norm': float(FLOW_STATE.get('psd_correction_norm', 0.0)) if FLOW_STATE.get('psd_correction_norm') is not None else None,
                    'schema_hash': schema_hash,
                    'seed': int(FLOW_STATE.get('seed', -1)),
                    'timestamp_local': ts
                }
            }
            out_path = RESULTS_DIR / f'{ts}_feasibility_report.json'
            with open(out_path, 'w', encoding='utf-8') as fh:
                _json.dump(artifact, fh, indent=2)
            print(f"[Flow 8] Wrote report: {out_path.name}")

            try:
                _np.savez(RESULTS_DIR / f'{ts}_samples.npz', **{f'm_{k}': _np.asarray(v) for k,v in FLOW_STATE['samples_X'].items()}, R=_np.asarray(R))
                print(f"[Flow 8] Saved samples: {ts}_samples.npz")
            except Exception:
                pass

            FLOW_STATE['report_path'] = str(out_path)
            FLOW_STATE['R'] = _np.asarray(R)
            FLOW_STATE['feasibility'] = feas
except Exception as e:
    print('Flow Step 6–8 error:', e)


[Flow 6] Distribution summaries:
  p_link: p50=7.600e-03, p10=4.945e-03, p90=1.174e-02
  p_swap: p50=8.938e-01, p10=8.471e-01, p90=9.313e-01
  R: p50=1.073e+03, p10=5.943e+02, p90=2.005e+03
  tau_c: p50=5.000e-04, p10=4.191e-04, p90=5.806e-04
[Flow 7] Feasibility:
  P(R ≥ 1e-06) = 100.000% (95% CI [99.923%, 100.000%])
  P(T2 ≥ 1.0·tau_c) = 100.000% (95% CI [99.923%, 100.000%])
[Flow 8] Wrote report: 20251109T192030_feasibility_report.json
[Flow 8] Saved samples: 20251109T192030_samples.npz


In [7]:
# 3) Schema Validation Helpers

import numpy as _np

def _broadcast(value, N, dtype=float):
    if isinstance(value, _np.ndarray):
        if value.shape[0] != N:
            raise ValueError(f"Length mismatch: {value.shape[0]} != {N}")
        return value.astype(dtype)
    return _np.full(N, value, dtype=dtype)

def validate_and_canonicalize_samples(X_dict):
    """Broadcast scalars to arrays; clamp within schema bounds; validate enums.
    Returns canonical dict of name->np.ndarray all length N.
    """
    if not isinstance(X_dict, dict):
        raise TypeError("X_dict must be dict")
    # Determine N from first array-like
    N = None
    for v in X_dict.values():
        try:
            N = len(v)
            break
        except Exception:
            continue
    if N is None:
        N = 1
    result = {}
    for name, meta in SCHEMA.items():
        if name in X_dict:
            raw = X_dict[name]
        else:
            raw = meta['default']
        dtype = float if meta['type'] in (float, int) else object
        arr = _broadcast(raw, N, dtype=dtype)
        # Clamp numeric
        if meta['type'] in (float, int):
            if 'min' in meta:
                arr = _np.maximum(arr, meta['min'])
            if 'max' in meta:
                arr = _np.minimum(arr, meta['max'])
        # Validate choices
        if meta['type'] is str and meta.get('choices') is not None:
            choices = set(meta['choices'])
            vals = set(arr.tolist())
            if not vals.issubset(choices):
                raise ValueError(f"Enum violation for {name}: {vals - choices}")
        result[name] = arr
    return result

print("Schema validation helpers loaded.")

Schema validation helpers loaded.


In [8]:
# 4) Rank-Correlation to PSD Correlation Matrix (tau_to_rho)
# rho = sin(pi * tau / 2); symmetricize; clip eigenvalues; renormalize diagonals

import numpy as _np

def tau_to_rho_matrix(tau_mat):
    tau = _np.asarray(tau_mat, dtype=float)
    if tau.ndim != 2 or tau.shape[0] != tau.shape[1]:
        raise ValueError("tau_to_rho_matrix expects a square matrix")
    rho = _np.sin(_np.pi * tau / 2.0)
    _np.fill_diagonal(rho, 1.0)
    rho = (rho + rho.T) / 2.0
    vals, vecs = _np.linalg.eigh(rho)
    clipped = _np.clip(vals, 1e-12, None)
    repair_norm = float(_np.linalg.norm(vals - clipped))
    if repair_norm > 0:
        rho = (vecs @ _np.diag(clipped) @ vecs.T)
        d = _np.sqrt(_np.clip(_np.diag(rho), 1e-12, None))
        rho = rho / _np.outer(d, d)
        rho = (rho + rho.T) / 2.0
    return rho, repair_norm

print("tau_to_rho_matrix ready (PSD-repaired correlation)")

tau_to_rho_matrix ready (PSD-repaired correlation)


In [9]:
# 5) Physics Kernels (CPU) for DLCZ Metrics
# Computes: losses, T, tau_c, p_link, p_swap, p_succ, R plus legacy comparisons; clips probabilities.

import numpy as _np

def _sanitize_arrays(alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber):
    def clamp(a, lo=None, hi=None):
        if lo is not None: a = _np.maximum(a, lo)
        if hi is not None: a = _np.minimum(a, hi)
        return a
    alpha = clamp(alpha, 0.0)
    L0_km = clamp(L0_km, 1e-9)
    eta_s = clamp(eta_s, 0.0, 1.0)
    eta_w = clamp(eta_w, 0.0, 1.0)
    eta_r = clamp(eta_r, 0.0, 1.0)
    eta_det = clamp(eta_det, 0.0, 1.0)
    V0 = clamp(V0, 0.0, 1.0)
    N_modes = clamp(N_modes, 1.0)
    kappa_power = clamp(kappa_power, 0.0)
    c_fiber = clamp(c_fiber, 1.0)
    n_swaps = _np.maximum(0, _np.rint(n_swaps).astype(int))
    return alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber

def _compute_metrics_cpu(X_dict):
    S = validate_and_canonicalize_samples(X_dict)
    N = len(next(iter(S.values())))
    alpha = S['alpha_dB_per_km'].astype(float)
    L0_km = S['L0_km'].astype(float)
    eta_s = S['eta_s'].astype(float)
    eta_w = S['eta_w'].astype(float)
    eta_r = S['eta_r'].astype(float)
    eta_det = S['eta_det'].astype(float)
    V0 = S['V0'].astype(float)
    N_modes = S['N_modes'].astype(float)
    n_swaps = S['n_swaps'].astype(float)
    kappa_power = S['kappa_power'].astype(float)
    c_fiber = S['c_fiber_mps'].astype(float)
    connector_loss_dB = S.get('connector_loss_dB').astype(float)
    n_connectors = S.get('n_connectors').astype(float)
    splice_loss_dB = S.get('splice_loss_dB').astype(float)
    n_splices = S.get('n_splices').astype(float)
    filter_loss_dB = S.get('filter_loss_dB').astype(float)
    p_dark = S.get('p_dark').astype(float)
    raman_rate_per_ns = S.get('raman_rate_per_ns').astype(float)
    gate_ns = S.get('gate_ns').astype(float)
    geometry = S.get('geometry')
    kappa_mode = S.get('kappa_mode')

    alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber = _sanitize_arrays(
        alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber
    )

    # Loss budget
    L_dB = alpha * L0_km + n_connectors * connector_loss_dB + n_splices * splice_loss_dB + filter_loss_dB
    # Central vs end geometry transmissivity handling
    T_effective = _np.empty(N, dtype=float)
    mask_central = _np.array([g == 'central_bsm' for g in geometry])
    mask_end = ~mask_central
    if mask_end.any():
        T_effective[mask_end] = 10.0 ** (-L_dB[mask_end] / 10.0)
    if mask_central.any():
        L_dB_arm = (alpha[mask_central] * (L0_km[mask_central] / 2.0) + (n_connectors[mask_central] * connector_loss_dB[mask_central] + n_splices[mask_central] * splice_loss_dB[mask_central] + filter_loss_dB[mask_central]) / 2.0)
        T_arm = 10.0 ** (-L_dB_arm / 10.0)
        T_effective[mask_central] = T_arm ** 2

    # Memory efficiency (write * read)
    eta_mem = eta_w * eta_r

    # Elementary link success (corrected) and false-herald suppression
    p_link = 0.5 * eta_s * eta_mem * (eta_det ** 2) * T_effective
    p_link = _np.clip(p_link, 1e-18, 1.0)
    p_false = 1.0 - _np.exp(-(p_dark + raman_rate_per_ns * gate_ns))
    p_link_eff = p_link * (1.0 - p_false)

    # Swap success depending on mode
    p_swap = _np.empty_like(p_link_eff)
    mask_power = _np.array([m == 'power' for m in kappa_mode])
    mask_lin = _np.array([m == 'linear' for m in kappa_mode])
    mask_sq = _np.array([m == 'square' for m in kappa_mode])
    if mask_power.any(): p_swap[mask_power] = V0[mask_power] ** kappa_power[mask_power]
    if mask_lin.any(): p_swap[mask_lin] = V0[mask_lin]
    if mask_sq.any(): p_swap[mask_sq] = V0[mask_sq] ** 2.0
    p_swap = _np.clip(p_swap, 1e-12, 1.0)

    # Overall success probability with swap chaining
    p_succ = p_link_eff * (p_swap ** n_swaps)
    p_succ = _np.clip(p_succ, 1e-24, 1.0)

    # Timing & throughput
    tau_c = (L0_km * 1e3 * 2.0) / _np.maximum(c_fiber, 1e-24)  # full round-trip
    R = (N_modes * p_succ) / _np.maximum(tau_c, 1e-24)

    # Legacy expressions (for comparison)
    p_link_legacy = 0.5 * eta_s * eta_mem * (eta_det ** 2) * 10.0 ** (-L_dB / 10.0)
    p_link_legacy = _np.clip(p_link_legacy, 1e-18, 1.0)
    p_swap_legacy = _np.clip(V0 ** kappa_power * (eta_r * eta_det) ** 2 * 0.5, 1e-18, 1.0)
    tau_oneway = (L0_km * 1e3) / _np.maximum(c_fiber, 1e-24)
    T_avg_legacy = tau_oneway / _np.maximum(N_modes * p_link_legacy, 1e-24) * (1.0 / _np.maximum(p_swap_legacy, 1e-24)) ** n_swaps
    R_legacy = 1.0 / _np.maximum(T_avg_legacy, 1e-30)

    return {
        'T_effective': T_effective,
        'tau_c': tau_c,
        'p_link': p_link_eff,
        'p_swap': p_swap,
        'p_succ': p_succ,
        'R': R,
        'p_link_legacy': p_link_legacy,
        'p_swap_legacy': p_swap_legacy,
        'R_legacy': R_legacy,
        'T_avg_legacy': T_avg_legacy,
    }

print("CPU physics kernels loaded.")

CPU physics kernels loaded.


In [10]:
# 6) Physics Kernels (Torch/Device) and Device-Aware Wrapper
# Uses float32 tensors; falls back to CPU if device unsupported.

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
except Exception:
    torch = None
    HAS_TORCH = False


def compute_metrics_torch(X_dict, device):
    if (not HAS_TORCH) or str(device).lower() in (None, 'cpu'):
        return _compute_metrics_cpu(X_dict)
    dev = torch.device(device)
    S = validate_and_canonicalize_samples(X_dict)
    N = len(next(iter(S.values())))
    # Helper to move/broadcast
    def tfill(x):
        return torch.full((N,), float(x), device=dev, dtype=torch.float32)

    # Scalars from medians of arrays (device path is illustrative; CPU path preferred)
    def med(name):
        a = S[name]
        return float(_np.median(a))

    alpha = med('alpha_dB_per_km')
    L0_km = med('L0_km')
    eta_s = med('eta_s')
    eta_w = med('eta_w')
    eta_r = med('eta_r')
    eta_det = med('eta_det')
    V0 = med('V0')
    N_modes = med('N_modes')
    n_swaps = int(round(med('n_swaps')))
    kappa_power = med('kappa_power')
    c_fiber = med('c_fiber_mps')
    connector_loss_dB = med('connector_loss_dB')
    n_connectors = int(round(med('n_connectors')))
    splice_loss_dB = med('splice_loss_dB')
    n_splices = int(round(med('n_splices')))
    filter_loss_dB = med('filter_loss_dB')

    alpha_t = tfill(alpha)
    L0_t = tfill(L0_km)
    eta_s_t = tfill(eta_s)
    eta_w_t = tfill(eta_w)
    eta_r_t = tfill(eta_r)
    eta_det_t = tfill(eta_det)
    V0_t = tfill(V0)
    N_modes_t = tfill(N_modes)
    n_swaps_t = tfill(n_swaps)
    kappa_power_t = tfill(kappa_power)
    c_fiber_t = tfill(c_fiber)

    L_dB = alpha_t * L0_t + n_connectors * connector_loss_dB + n_splices * splice_loss_dB + filter_loss_dB
    T_full = torch.pow(10.0, -L_dB / 10.0)

    eta_mem_t = eta_w_t * eta_r_t

    p_link_new = 0.5 * eta_s_t * eta_mem_t * (eta_det_t ** 2) * T_full
    p_link_new = torch.clamp(p_link_new, 1e-18, 1.0)

    p_swap_new = torch.pow(V0_t, kappa_power_t)
    p_swap_new = torch.clamp(p_swap_new, 1e-12, 1.0)

    p_succ = p_link_new * torch.pow(p_swap_new, n_swaps_t)
    p_succ = torch.clamp(p_succ, 1e-24, 1.0)

    tau_c = (L0_t * 1e3 * 2.0) / torch.clamp(c_fiber_t, 1e-24)
    R = (N_modes_t * p_succ) / torch.clamp(tau_c, 1e-24)

    return {
        'R': R.cpu().numpy(),
    }


def run_on_device(cfg_df, sampleN, device_str='cpu'):
    """Sample marginals from a config DataFrame and run kernels on selected device.
    Returns (sampled_dict, metrics_dict, elapsed_seconds).
    """
    import time as _time
    sampled = {}
    rows = cfg_df.to_dict(orient='records')
    for row in rows:
        key = row.get('variable') or row.get('name') or row.get('key')
        sampled[key] = sample_marginal_inline(row, sampleN, rng=RNG)
    start = _time.time()
    if (not HAS_TORCH) or device_str in (None, 'cpu', 'numpy'):
        metrics = _compute_metrics_cpu(sampled)
    else:
        try:
            metrics = compute_metrics_torch(sampled, device_str)
        except Exception as e:
            print('Device path failed -> CPU fallback. Error:', e)
            metrics = _compute_metrics_cpu(sampled)
    elapsed = _time.time() - start
    return sampled, metrics, elapsed

print(f"Torch available: {HAS_TORCH}")

Torch available: False


In [11]:
# 7) Inline Marginal Sampler (const/normal/lognormal/uniform/beta)

import numpy as _np

def sample_marginal_inline(row, N, rng=None):
    if rng is None:
        rng = _np.random.default_rng()
    if not isinstance(row, dict):
        raise TypeError("row must be dict with keys: variable/dist/params")
    dist = (row.get('dist') or '').lower()
    params = row.get('params') or {}
    if dist == 'const':
        val = float(params.get('value', 0.0))
        return _np.full(N, val, dtype=float)
    if dist == 'normal':
        mu = float(params.get('mu', 0.0)); sigma = max(1e-12, float(params.get('sigma', 1.0)))
        return rng.normal(mu, sigma, size=N)
    if dist == 'lognormal':
        mu = float(params.get('mu', 0.0)); sigma = max(1e-12, float(params.get('sigma', 1.0)))
        return rng.lognormal(mean=mu, sigma=sigma, size=N)
    if dist == 'uniform':
        low = float(params.get('low', 0.0)); high = float(params.get('high', 1.0))
        if high < low: low, high = high, low
        return rng.uniform(low, high, size=N)
    if dist == 'beta':
        a = float(params.get('a', 1.0)); b = float(params.get('b', 1.0))
        low = float(params.get('low', 0.0)); high = float(params.get('high', 1.0))
        raw = rng.beta(a, b, size=N)
        return low + (high - low) * raw
    raise ValueError(f"Unsupported dist family: {dist}")

print("Marginal sampler ready.")

Marginal sampler ready.


In [12]:
# 8) Backend Detection (CUDA/DirectML/CPU) and Guidance Printers

import subprocess, shutil, platform

def detect_hardware():
    hw = {
        'nvidia_smi': False,
        'nvidia_gpus': [],
        'torch': HAS_TORCH,
        'cuda_available': False,
        'dml': False,
    }
    try:
        res = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True, check=False)
        if res.returncode == 0 and res.stdout.strip():
            hw['nvidia_smi'] = True
            hw['nvidia_gpus'] = [l.strip() for l in res.stdout.splitlines() if l.strip()]
    except Exception:
        pass
    if HAS_TORCH:
        try:
            hw['cuda_available'] = bool(torch.cuda.is_available())
        except Exception:
            hw['cuda_available'] = False
    try:
        import torch_directml  # noqa: F401
        hw['dml'] = True
    except Exception:
        hw['dml'] = False
    return hw


def print_conda_cuda_instructions(env_name='qnet-cuda'):
    print('\nConda steps for CUDA-enabled env:')
    print(f'  conda create -n {env_name} python=3.11 -y')
    print(f'  conda activate {env_name}')
    print('  conda install -y -c pytorch -c nvidia pytorch torchvision torchaudio pytorch-cuda=12.1')
    print('Start Jupyter from that env and re-open this notebook.')


def print_dml_venv_instructions(venv_path='venvs/qnet-dml'):
    print('\nWindows venv steps for DirectML:')
    print(f'  python -m venv {venv_path}')
    print(f'  {venv_path}\\Scripts\\activate')
    print('  python -m pip install --upgrade pip setuptools wheel')
    print('  python -m pip install torch-directml')
    print('Start Jupyter from that venv and re-open this notebook.')

print('Hardware detection helpers ready (CPU default).')

Hardware detection helpers ready (CPU default).


In [13]:
# 9) Backend Selection UI (ipywidgets)
# Dropdowns for backend and device; Apply button sets NOTEBOOK_BACKEND/SELECTED_DEVICE

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _BE_widgets = globals().get('_backend_select_widgets')
    if _BE_widgets:
        for w in _BE_widgets:
            try: w.close()
            except Exception: pass
    _BE_widgets = []

    hw = detect_hardware()
    options = [('CPU','cpu')]
    if hw.get('cuda_available') or hw.get('nvidia_gpus'):
        options.insert(0, ('CUDA (GPU)','cuda'))
    if hw.get('dml'):
        options.insert(1, ('DirectML (GPU)','dml'))

    backend_dd = widgets.Dropdown(options=options, value='cpu', description='Backend:')
    device_dd = widgets.Dropdown(options=[('CPU','cpu')], description='Device:')
    status = widgets.HTML(value='')
    apply_btn = widgets.Button(description='Apply', button_style='success')

    def _on_backend_change(ch):
        if ch['name'] == 'value':
            if ch['new'] == 'cuda' and hw.get('nvidia_gpus'):
                device_dd.options = [(f'CUDA {i}: {name}', f'cuda:{i}') for i, name in enumerate(hw['nvidia_gpus'])] or [('CUDA:0','cuda:0')]
                device_dd.value = device_dd.options[0][1]
            elif ch['new'] == 'dml':
                device_dd.options = [('DirectML','dml')]
                device_dd.value = 'dml'
            else:
                device_dd.options = [('CPU','cpu')]
                device_dd.value = 'cpu'
            status.value = ''

    backend_dd.observe(_on_backend_change, names='value')

    def _on_apply(_):
        global NOTEBOOK_BACKEND, SELECTED_DEVICE
        NOTEBOOK_BACKEND = backend_dd.value
        SELECTED_DEVICE = device_dd.value
        status.value = f"<span style='color:#5cb85c'>Selected: <b>{NOTEBOOK_BACKEND}</b> on <b>{SELECTED_DEVICE}</b></span>"

    apply_btn.on_click(_on_apply)

    panel = widgets.VBox([
        widgets.HTML("<b>Backend selection</b>"),
        widgets.HBox([backend_dd, device_dd, apply_btn]),
        widgets.HTML("<div class='small-hint'>CPU is recommended unless you have a configured GPU env.</div>"),
        status
    ])
    display(panel)
    _BE_widgets.extend([backend_dd, device_dd, apply_btn, status, panel])
    globals()['_backend_select_widgets'] = _BE_widgets
except Exception as e:
    print('Backend UI unavailable:', e)


In [ ]:
# Marginal & Copula Configuration (Strict Ordering)
# This cell MUST be executed before Flow 0–3.
# It sets WORKING_MARGINALS and any copula kendall_tau_pairs in CONFIG.

try:
    import json as _json
    import numpy as _np
    import ipywidgets as widgets

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    # Initialize WORKING_MARGINALS if absent
    if 'WORKING_MARGINALS' not in globals():
        WORKING_MARGINALS = {
            'eta_s': {'family':'beta','params':{'a':70,'b':30},'unit':'frac'},
            'eta_det': {'family':'beta','params':{'a':75,'b':25},'unit':'frac'},
            'alpha_dB_per_km': {'family':'lognormal','params':{'mu_log':_np.log(SCHEMA['alpha_dB_per_km']['default']), 'sigma_log':0.05},'unit':'dB/km'},
            'L0_km': {'family':'uniform','params':{'low':SCHEMA['L0_km']['default']*0.9,'high':SCHEMA['L0_km']['default']*1.1},'unit':'km'},
            'V0': {'family':'beta','params':{'a':90,'b':10},'unit':'vis'},
            'T2_ms': {'family':'lognormal','params':{'mu_log':_np.log(max(SCHEMA['T2_ms']['default'],1e-9)),'sigma_log':0.1},'unit':'ms'},
            'N_modes': {'family':'uniform','params':{'low':SCHEMA['N_modes']['default']*0.95,'high':SCHEMA['N_modes']['default']*1.05},'unit':'count'}
        }

    # UI elements
    status_msg = widgets.HTML('')
    var_select = widgets.Select(options=list(WORKING_MARGINALS.keys()), value=list(WORKING_MARGINALS.keys())[0], description='variable')
    family_dd = widgets.Dropdown(options=['beta','lognormal','uniform','truncnorm'], value=WORKING_MARGINALS[var_select.value]['family'], description='family:')
    params_box = widgets.VBox([])
    save_btn = widgets.Button(description='Apply', button_style='success')

    def _rebuild_params(var):
        spec = WORKING_MARGINALS[var]
        fam = spec['family']
        params = spec.get('params', {})
        widgets_list = []
        if fam=='beta':
            a_w = widgets.FloatText(value=params.get('a',2.0), description='a')
            b_w = widgets.FloatText(value=params.get('b',5.0), description='b')
            widgets_list=[a_w,b_w]
        elif fam=='lognormal':
            mu_w = widgets.FloatText(value=params.get('mu_log',0.0), description='mu_log')
            sg_w = widgets.FloatText(value=params.get('sigma_log',0.1), description='sigma_log')
            widgets_list=[mu_w,sg_w]
        elif fam=='uniform':
            low_w = widgets.FloatText(value=params.get('low',0.0), description='low')
            high_w = widgets.FloatText(value=params.get('high',1.0), description='high')
            widgets_list=[low_w,high_w]
        elif fam in ('truncnorm','truncated_normal'):
            low_w = widgets.FloatText(value=params.get('low',0.0), description='low')
            high_w = widgets.FloatText(value=params.get('high',1.0), description='high')
            loc_w = widgets.FloatText(value=params.get('loc',0.0), description='loc')
            sc_w = widgets.FloatText(value=params.get('scale',1.0), description='scale')
            widgets_list=[low_w,high_w,loc_w,sc_w]
        params_box.children = widgets_list

    def _on_var(change):
        family_dd.value = WORKING_MARGINALS[change['new']]['family']
        _rebuild_params(change['new'])

    def _on_family(change):
        WORKING_MARGINALS[var_select.value]['family']=change['new']
        _rebuild_params(var_select.value)

    def _on_apply(_):
        # Harvest params
        fam = family_dd.value
        harvested={}
        for w in params_box.children:
            harvested[w.description]=w.value
        WORKING_MARGINALS[var_select.value]={'family':fam,'params':harvested,'unit':WORKING_MARGINALS[var_select.value].get('unit','')}
        status_msg.value = "<span style='color:#5cb85c'>Updated marginal for %s</span>" % var_select.value
        FLOW_STATE['configured']=True

    var_select.observe(_on_var, names='value')
    family_dd.observe(_on_family, names='value')
    save_btn.on_click(_on_apply)

    _rebuild_params(var_select.value)

    display(widgets.VBox([
        widgets.HTML('<b>Strict Configuration: Set marginals before running Flow 0–3</b>'),
        var_select,
        family_dd,
        params_box,
        save_btn,
        status_msg,
        widgets.HTML('<i>After adjusting all desired marginals, run the Flow 0–3 cell.</i>')
    ]))
except Exception as e:
    print('Strict configuration UI error:', e)


Output()

In [15]:
# 12) Single-Run Inline/Device Runner and Visual Diagnostics
# Loads config (JSON/CSV), chooses inline vs device path, saves samples+metrics, visualizes distributions.

import json as _json, os as _os
try:
    import pandas as _pd
    _HAS_PANDAS = True
except Exception:
    _HAS_PANDAS = False

try:
    import matplotlib.pyplot as _plt
    _HAS_MPL = True
except Exception:
    _HAS_MPL = False

_DEF_MIN_SAMPLES = 10000
_DEF_MAX_SAMPLES = int(1e7)


def _brief_stats(x):
    return dict(min=float(_np.min(x)), p5=float(_np.percentile(x,5)), median=float(_np.median(x)), p95=float(_np.percentile(x,95)), max=float(_np.max(x)), mean=float(_np.mean(x)))


def _viz_arrays(named_arrays, title_prefix, max_points=50000, cols=3):
    if not _HAS_MPL:
        print('Matplotlib not available for arrays visualization.')
        return
    items = [(k,v) for k,v in named_arrays.items() if isinstance(v, _np.ndarray)]
    if not items:
        print('No arrays to visualize.')
        return
    N = len(items[0][1])
    if N > max_points:
        rng = _np.random.default_rng(123)
        idx = rng.choice(N, size=max_points, replace=False)
    else:
        idx = slice(None)
    rows = (len(items)+cols-1)//cols
    fig, axes = _plt.subplots(rows, cols, figsize=(cols*5.0, rows*3.5))
    axes = _np.atleast_1d(axes).reshape(rows, cols)
    for i,(k,a) in enumerate(items):
        a = a[idx]
        r = i//cols; c = i%cols
        ax = axes[r,c]
        bins = 90 if len(a) > 4000 else max(30, int(len(a)//70))
        ax.hist(a, bins=bins, color='#4C72B0', alpha=0.85)
        st = _brief_stats(a)
        ax.set_title(f"{k}\nmin={st['min']:.3e} med={st['median']:.3e} max={st['max']:.3e}")
        if _np.all(a>0) and _np.max(a)/max(1e-30,_np.min(a)) > 1e3:
            ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
    for j in range(i+1, rows*cols):
        axes[j//cols, j%cols].axis('off')
    fig.suptitle(f'{title_prefix} histograms')
    fig.tight_layout(); _plt.show()


def _viz_metrics(metrics, max_points=50000, cols=3):
    if not _HAS_MPL:
        print('Matplotlib not available for metrics visualization.')
        return
    arr_items = [(k,v) for k,v in metrics.items() if isinstance(v, _np.ndarray)]
    if not arr_items:
        print('No metric arrays to visualize.')
        return
    N = len(arr_items[0][1])
    if N > max_points:
        rng = _np.random.default_rng(321)
        idx = rng.choice(N, size=max_points, replace=False)
    else:
        idx = slice(None)
    rows = (len(arr_items)+cols-1)//cols
    fig, axes = _plt.subplots(rows, cols, figsize=(cols*5.0, rows*3.5))
    axes = _np.atleast_1d(axes).reshape(rows, cols)
    for i,(k,a) in enumerate(arr_items):
        a = a[idx]
        r=i//cols; c=i%cols
        ax = axes[r,c]
        bins = 90 if len(a)>4000 else max(30,int(len(a)//70))
        ax.hist(a, bins=bins, color='#55A868', alpha=0.85)
        st = _brief_stats(a)
        ax.set_title(f"{k}\nmin={st['min']:.3e} med={st['median']:.3e} max={st['max']:.3e}")
        if _np.all(a>0) and _np.max(a)/max(1e-30,_np.min(a))>1e3:
            ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
    for j in range(i+1, rows*cols):
        axes[j//cols, j%cols].axis('off')
    fig.suptitle('Metrics histograms'); fig.tight_layout(); _plt.show()
    if 'R' in metrics and isinstance(metrics['R'], _np.ndarray):
        r = metrics['R'][idx]
        r_sorted = _np.sort(r); y = _np.linspace(0,1,len(r_sorted), endpoint=False)
        _plt.figure(figsize=(6,4))
        _plt.plot(r_sorted, y, lw=2)
        _plt.xlabel('R'); _plt.ylabel('CDF'); _plt.title('CDF of R'); _plt.grid(True, alpha=0.3)
        if _np.all(r_sorted>0) and r_sorted[-1]/max(1e-30,r_sorted[0])>1e3:
            _plt.xscale('log')
        _plt.show()
    if 'R' in metrics and 'R_legacy' in metrics:
        R = metrics['R'][idx]; Rl = metrics['R_legacy'][idx]
        _plt.figure(figsize=(5,5))
        _plt.scatter(Rl, R, s=5, alpha=0.3)
        lim = [min(_np.min(Rl), _np.min(R)), max(_np.max(Rl), _np.max(R))]
        lim[0] = max(lim[0], 1e-30); lim[1] = max(lim[1], lim[0]*1.1)
        _plt.plot(lim, lim, 'r--', lw=1)
        _plt.xscale('log'); _plt.yscale('log')
        _plt.xlabel('R_legacy'); _plt.ylabel('R (new)'); _plt.title('R new vs R legacy')
        _plt.grid(True, alpha=0.3)
        _plt.show()


def _load_config_candidates():
    candidates = [CONFIG_DIR/'marginal_config.json', CONFIG_DIR/'marginal_catalog.csv', Path('marginal_config.json'), Path('marginal_catalog.csv')]
    seen = []
    for p in candidates:
        p = Path(p)
        if p in seen: continue
        seen.append(p)
        if p.exists():
            yield p


def _load_config():
    cfg_df = None; cfg_raw = None
    for p in _load_config_candidates():
        try:
            if p.suffix.lower()=='.csv':
                if not _HAS_PANDAS: continue
                df = _pd.read_csv(p)
                if df is not None and not df.empty:
                    cfg_df = df; break
            else:
                raw = _json.load(open(p,'r',encoding='utf-8'))
                if isinstance(raw, dict) and 'marginals' in raw:
                    rows = [{'variable':m.get('variable') or m.get('name'), 'dist':m.get('dist'), 'params':m.get('params')} for m in raw['marginals']]
                    cfg_df = _pd.DataFrame(rows) if _HAS_PANDAS else None
                    cfg_raw = rows if not _HAS_PANDAS else None
                    break
                elif isinstance(raw, list):
                    cfg_df = _pd.DataFrame(raw) if _HAS_PANDAS else None
                    cfg_raw = raw if not _HAS_PANDAS else None
                    break
        except Exception as e:
            print('Failed to parse', p, '->', e)
    if cfg_df is None and cfg_raw is None and 'WORKING_MARGINALS' in globals():
        rows = [{'variable':k,'dist':v['dist'],'params':v.get('params',{})} for k,v in WORKING_MARGINALS.items()]
        cfg_df = _pd.DataFrame(rows) if _HAS_PANDAS else None
        cfg_raw = rows if not _HAS_PANDAS else None
    return cfg_df, cfg_raw


def run_once(backend: str, samples: int):
    timestamp = time.strftime('%Y%m%dT%H%M%S')
    run_name = f'run_{timestamp}'
    summary_path = RESULTS_DIR / f'{run_name}_summary.json'
    samples_path = RESULTS_DIR / f'{run_name}_samples.npz'

    cfg_df, cfg_raw = _load_config()
    sample_fn = sample_marginal_inline
    prefer_inline = backend in ('cpu','numpy') and callable(sample_fn)

    if (cfg_df is not None or cfg_raw is not None) and prefer_inline:
        sampled = {}
        rows = cfg_df.to_dict(orient='records') if cfg_df is not None else cfg_raw
        for r in rows:
            key = r.get('variable')
            sampled[key] = sample_fn(r, samples, rng=RNG)
        _viz_arrays(sampled, 'Sampled inputs')
        metrics = _compute_metrics_cpu(sampled)
    elif (cfg_df is not None) and callable(run_on_device):
        sampled, metrics, _ = run_on_device(cfg_df, sampleN=samples, device_str=backend)
        _viz_arrays(sampled, 'Sampled inputs')
    else:
        print('No usable config. Create marginals first.')
        return

    _viz_metrics(metrics)
    summary = {'run_name': run_name, 'backend': backend, 'sampleN': samples, 'metrics_keys': list(metrics.keys())}
    with open(summary_path,'w',encoding='utf-8') as fh:
        _json.dump(summary, fh, indent=2)
    try:
        _np.savez_compressed(samples_path, **sampled, **{k:v for k,v in metrics.items() if isinstance(v,_np.ndarray)})
    except Exception as e:
        print('Warning: could not save samples ->', e)
    print('Run finished. Summary:', summary_path, 'Samples+metrics:', samples_path)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    backend_dd = widgets.Dropdown(options=[('CPU','cpu'),('NumPy','numpy'),('CUDA','cuda'),('DirectML','dml')], value='cpu', description='Backend:')
    sample_txt = widgets.Text(value='', placeholder=f'>={_DEF_MIN_SAMPLES}', description='Samples:')
    run_btn = widgets.Button(description='Run', button_style='success')
    out = widgets.Output()

    def _parse_samples(s):
        s = (s or '').strip()
        if not s: return _DEF_MIN_SAMPLES
        try: n = int(s)
        except Exception: return _DEF_MIN_SAMPLES
        if n < _DEF_MIN_SAMPLES: n = _DEF_MIN_SAMPLES
        if n > _DEF_MAX_SAMPLES: n = _DEF_MAX_SAMPLES
        return n

    def _on_run(_):
        with out:
            clear_output(wait=True)
            backend = backend_dd.value
            samples = _parse_samples(sample_txt.value)
            print(f'Backend={backend} Samples={samples}')
            run_once(backend, samples)

    run_btn.on_click(_on_run)
    display(widgets.VBox([widgets.HBox([backend_dd, sample_txt, run_btn]), out]))
except Exception as e:
    print('Run UI unavailable; fallback to text prompts.', e)


In [16]:
# Results Visualization UI (histograms, CDF, optional KDE, log-scale)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import re, json as _json
    from pathlib import Path as _Path
    import numpy as _np
    import matplotlib.pyplot as _plt
    from scipy.stats import gaussian_kde as _kde

    results_dir = RESULTS_DIR
    npz_files = sorted(results_dir.glob('*.npz'), key=lambda p: p.stat().st_mtime)
    json_files = sorted(results_dir.glob('*_summary.json'), key=lambda p: p.stat().st_mtime)

    npz_dd = widgets.Dropdown(options=[(p.name, str(p)) for p in npz_files] or [('(none)','')], description='Samples:')
    json_dd = widgets.Dropdown(options=[(p.name, str(p)) for p in json_files] or [('(none)','')], description='Summary:')
    refresh_btn = widgets.Button(description='Refresh list')
    plot_btn = widgets.Button(description='Plot', button_style='success')
    kde_chk = widgets.Checkbox(value=False, description='KDE overlay')
    log_chk = widgets.Checkbox(value=False, description='Log X')
    reconstruct_chk = widgets.Checkbox(value=True, description='Reconstruct R if missing')
    status = widgets.HTML(value='')
    out = widgets.Output()

    def _refresh(_=None):
        # Recompute file lists locally; update dropdowns without nonlocal reassignment
        _npz_files = sorted(results_dir.glob('*.npz'), key=lambda p: p.stat().st_mtime)
        _json_files = sorted(results_dir.glob('*_summary.json'), key=lambda p: p.stat().st_mtime)
        npz_dd.options = [(p.name, str(p)) for p in _npz_files] or [('(none)','')]
        json_dd.options = [(p.name, str(p)) for p in _json_files] or [('(none)','')]
        status.value = "<span style='color:#5cb85c'>Refreshed.</span>"

    def _load_npz(path):
        if not path: return None
        try:
            return _np.load(_Path(path), allow_pickle=True)
        except Exception as e:
            status.value = f"<span style='color:#d9534f'>Failed to load npz: {e}</span>"
            return None

    def _try_reconstruct_R(data):
        if 'R' in data: return _np.asarray(data['R'])
        files = list(data.files)
        m_keys = [k for k in files if k.startswith('m_')]
        name_keys = ['marginal_names','var_names','columns','names','m_names','marginal_keys']
        var_names = None
        for nk in name_keys:
            if nk in files:
                arr = data[nk]
                if isinstance(arr,_np.ndarray) and arr.dtype==object: arr = arr.tolist()
                var_names = list(arr); break
        if m_keys and var_names:
            sampled = {}
            for mk in m_keys:
                key = mk[2:]
                if key in var_names:
                    sampled[key] = _np.asarray(data[mk])
            try:
                metrics = _compute_metrics_cpu(sampled)
                if 'R' in metrics: return _np.asarray(metrics['R'])
            except Exception:
                pass
        return None

    def _plot(_=None):
        with out:
            clear_output(wait=True)
            p = npz_dd.value
            data = _load_npz(p) if p else None
            R = None
            if data is not None:
                if 'R' in data:
                    R = _np.asarray(data['R'])
                elif reconstruct_chk.value:
                    R = _try_reconstruct_R(data)
            if R is None:
                print('No R found; nothing to plot.')
                return
            n = len(R); mean = _np.mean(R); median = _np.median(R)
            fig, axes = _plt.subplots(1,2, figsize=(11,4))
            bins = 120 if n>10000 else max(40, int(n//80))
            axes[0].hist(R, bins=bins, color='#55A868', alpha=0.8)
            if kde_chk.value:
                try:
                    xs = _np.linspace(R.min(), R.max(), 300)
                    kde = _kde(R)
                    axes[0].plot(xs, kde(xs)*(R.max()-R.min())/bins, color='black', lw=1.2)
                except Exception:
                    pass
            axes[0].set_title(f"Histogram R\nmean={mean:.3e} median={median:.3e}")
            if log_chk.value and _np.all(R>0):
                axes[0].set_xscale('log')
            axes[0].grid(True, alpha=0.3)
            axes[1].plot(_np.sort(R), _np.linspace(0,1,n), lw=2, color='#4C72B0')
            axes[1].set_title('Empirical CDF of R')
            axes[1].grid(True, alpha=0.3)
            fig.tight_layout(); _plt.show()

    refresh_btn.on_click(_refresh)
    plot_btn.on_click(_plot)
    panel = widgets.VBox([
        widgets.HBox([npz_dd, json_dd]),
        widgets.HBox([reconstruct_chk, kde_chk, log_chk]),
        widgets.HBox([refresh_btn, plot_btn]),
        status, out
    ])
    display(panel)
except Exception as e:
    print('Results UI unavailable:', e)


In [17]:
# 14) Feasibility Report UI (R_target thresholding, reconstruction path)
# Simplified: remove datetime usage; rely on local computer clock via time module.
# Filename now uses epoch seconds; report includes 'generated_epoch' and 'generated_local' (localtime).

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import json as _json
    import numpy as _np
    import time as _time

    r_target_txt = widgets.FloatText(value=1e-6, description='R target:')
    regen_btn = widgets.Button(description='Generate report', button_style='success')
    reconstruct_chk2 = widgets.Checkbox(value=True, description='Reconstruct R if needed')
    status2 = widgets.HTML(value='')
    report_out = widgets.Output()

    def _load_R_from_npz(path):
        try:
            data = _np.load(path, allow_pickle=True)
        except Exception as e:
            status2.value = f"<span style='color:#d9534f'>Load failed: {e}</span>"; return None
        if 'R' in data:
            return _np.asarray(data['R'])
        if reconstruct_chk2.value:
            files = list(data.files)
            m_keys = [k for k in files if k.startswith('m_')]
            sampled = {mk[2:]: _np.asarray(data[mk]) for mk in m_keys}
            try:
                metrics = _compute_metrics_cpu(sampled)
                if 'R' in metrics:
                    return _np.asarray(metrics['R'])
            except Exception:
                pass
        return None

    def _generate(_):
        with report_out:
            clear_output(wait=True)
            R_target = float(r_target_txt.value)
            files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if not files:
                print('No results found.'); status2.value = '<span style="color:#d9534f">No results.</span>'; return
            latest = files[-1]
            R = _load_R_from_npz(latest)
            epoch = int(_time.time())
            local_str = _time.strftime('%Y-%m-%d %H:%M:%S')
            report = {'generated_epoch': epoch, 'generated_local': local_str, 'R_target': R_target, 'source': latest.name}
            if R is None:
                report['R_present'] = False
                report['verdict'] = 'No R metric available.'
            else:
                R = _np.asarray(R)
                report['R_present'] = True
                report['n_samples'] = int(len(R))
                report['mean_R'] = float(_np.mean(R))
                report['median_R'] = float(_np.median(R))
                p10, p90 = _np.percentile(R, [10, 90])
                report['p10_R'] = float(p10); report['p90_R'] = float(p90)
                frac_above = float((R >= R_target).mean())
                report['frac_above_R_target'] = frac_above
                if frac_above >= 0.5:
                    verdict = 'Likely feasible'
                elif frac_above > 0.0:
                    verdict = 'Partially feasible'
                else:
                    verdict = 'Likely not feasible'
                report['verdict'] = verdict
            print(_json.dumps(report, indent=2))
            fname = RESULTS_DIR / (f'{epoch}_feasibility_report.json')
            with open(fname, 'w', encoding='utf-8') as f:
                _json.dump(report, f, indent=2)
            status2.value = f"<span style='color:#5cb85c'>Report saved to {fname.name}</span>"

    regen_btn.on_click(_generate)
    panel2 = widgets.VBox([
        widgets.HTML('<b>Feasibility report</b>'),
        widgets.HBox([r_target_txt, reconstruct_chk2, regen_btn]),
        status2,
        report_out
    ])
    display(panel2)
except Exception as e:
    print('Feasibility UI unavailable:', e)

In [18]:
# 15) Detailed Feasibility Report (Plotly histogram + ECDF, executive summary)
# Idempotent rendering avoiding duplicates; toggle best sample; JSON export; sensitivity proxy.

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    import json as _json
    import numpy as _np
    try:
        import plotly.graph_objects as go
        _HAS_PLOTLY_DET = True
    except Exception:
        _HAS_PLOTLY_DET = False

    # Cleanup prior detailed report widgets
    _DRW = globals().get('_detailed_report_widgets')
    if _DRW:
        for w in _DRW:
            try: w.close()
            except Exception: pass
    _DRW = []

    def _register(*ws):
        for w in ws: _DRW.append(w)
        globals()['_detailed_report_widgets'] = _DRW

    def _extract_R(report_dict):
        for k in ('R','R_array','R_samples'):
            if isinstance(report_dict.get(k), (list,_np.ndarray)):
                arr = _np.asarray(report_dict[k], dtype=float)
                if arr.size: return arr
        # Fallback look for metrics
        m = report_dict.get('metrics')
        if isinstance(m, dict) and 'R' in m:
            arr = _np.asarray(m['R'], dtype=float)
            if arr.size: return arr
        return None

    def display_detailed_report(report=None, R_target=None):
        if report is None:
            # Attempt to find latest feasibility report
            files = sorted(RESULTS_DIR.glob('*_feasibility_report.json'), key=lambda p: p.stat().st_mtime)
            if files:
                try:
                    report = _json.load(open(files[-1],'r',encoding='utf-8'))
                except Exception:
                    report = None
        if report is None:
            display(HTML('<b>No report found.</b>'))
            return
        R_arr = None
        if report.get('R_present') and 'frac_above_R_target' in report and 'mean_R' in report:
            # Only summary stored, try reconstruction via latest samples
            files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if files:
                data = _np.load(files[-1], allow_pickle=True)
                if 'R' in data:
                    R_arr = _np.asarray(data['R'])
        if R_arr is None:
            R_arr = _extract_R(report)
        if R_arr is None or R_arr.size == 0 or not _np.isfinite(R_arr).any():
            display(HTML('<b>Unable to obtain R array for detailed report.</b>'))
            return
        R_arr = _np.nan_to_num(R_arr, nan=0.0, posinf=0.0, neginf=0.0)
        if R_target is None:
            R_target = report.get('R_target', 1e-6)
        R_target = float(R_target)

        mean_R = float(_np.mean(R_arr)); median_R = float(_np.median(R_arr))
        p10, p90 = _np.percentile(R_arr, [10, 90])
        frac_above = float(_np.mean(R_arr >= R_target))
        best_idx = int(_np.argmax(R_arr)); best_R = float(R_arr[best_idx])
        if frac_above >= 0.5:
            verdict, color = 'Feasible', '#2ca02c'
        elif frac_above >= 0.1:
            verdict, color = 'Marginal', '#ff7f0e'
        else:
            verdict, color = 'Not feasible', '#d62728'

        header_html = (
            "<div style='border:1px solid #ddd;padding:10px;border-radius:6px;background:#fafafa'>"
            f"<h3 style='margin:0'>Detailed Feasibility Report</h3>"
            f"<div style='margin-top:6px'><span style='padding:6px 10px;background:{color};color:#fff;border-radius:4px;font-weight:600'>{verdict}</span>"
            f" <span style='margin-left:12px;color:#333;'>R_target = <b>{R_target:.3g}</b></span></div></div>"
        )

        metrics_html = "<table style='width:100%;border-collapse:collapse;margin-top:8px'>" \
            + f"<tr><td><b>Mean R</b></td><td>{mean_R:.4g}</td><td><b>Median R</b></td><td>{median_R:.4g}</td></tr>" \
            + f"<tr><td><b>10th %</b></td><td>{p10:.4g}</td><td><b>90th %</b></td><td>{p90:.4g}</td></tr>" \
            + f"<tr><td><b>Frac ≥ target</b></td><td>{frac_above:.3%}</td><td><b>Best R</b></td><td>{best_R:.4g}</td></tr>" \
            + "</table>"

        plot_out = widgets.Output()
        with plot_out:
            if _HAS_PLOTLY_DET:
                # Build Plotly figures
                n = int(R_arr.size)
                nbins = max(30, min(200, n // 80 if n > 0 else 30))
                base_h = go.Figure()
                base_h.add_trace(go.Histogram(x=R_arr, nbinsx=nbins, marker_color='#1f77b4', opacity=0.85))
                base_h.add_vline(x=R_target, line=dict(color='red', dash='dash'))
                base_h.update_layout(width=520, height=280, title='R histogram')
                base_c = go.Figure()
                sorted_R = _np.sort(R_arr)
                cdf = _np.arange(1,len(sorted_R)+1)/float(len(sorted_R))
                base_c.add_trace(go.Scatter(x=sorted_R, y=cdf, mode='lines', line=dict(color='#2ca02c')))
                base_c.add_vline(x=R_target, line=dict(color='red', dash='dash'))
                base_c.update_layout(width=520, height=280, title='R ECDF')
                # Use FigureWidget to render reliably in VS Code/Notebook without injecting HTML
                try:
                    fw_h = go.FigureWidget(base_h)
                    fw_c = go.FigureWidget(base_c)
                    display(widgets.HBox([
                        widgets.VBox([widgets.HTML('<b>Distribution</b>'), fw_h]),
                        widgets.VBox([widgets.HTML('<b>ECDF</b>'), fw_c])
                    ]))
                except Exception:
                    # Fallback to direct display if FigureWidget not available
                    display(base_h)
                    display(base_c)
            else:
                print('Plotly not available; install plotly for interactive figures.')

        best_toggle = widgets.ToggleButton(value=False, description='Show best sample')
        best_out = widgets.Output()
        save_btn = widgets.Button(description='Export JSON')
        save_msg = widgets.HTML()

        # Sensitivity proxy: rank correlation (Spearman approx via rank correlation formula)
        sens_out = widgets.Output()
        with sens_out:
            lines = ["<div style='font-size:90%;color:#333'>"]
            # Attempt to reconstruct sample dict if available
            # This is heuristic: look for most recent npz
            npz_files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if npz_files:
                try:
                    data = _np.load(npz_files[-1], allow_pickle=True)
                    sample_vars = [k for k in data.files if k.startswith('m_')]
                    for sv in sample_vars:
                        arr = _np.asarray(data[sv])
                        if arr.size == R_arr.size:
                            try:
                                r_corr = _np.corrcoef(_np.argsort(arr), _np.argsort(R_arr))[0,1]
                                if _np.isfinite(r_corr):
                                    lines.append(f"<div><b>{sv[2:]}</b>: sensitivity proxy = {r_corr:.3f}</div>")
                            except Exception:
                                pass
                except Exception:
                    pass
            lines.append("</div>")
            display(HTML("".join(lines)))

        def _on_best(change):
            if change['name']=='value':
                best_out.clear_output()
                if change['new']:
                    with best_out:
                        idx = best_idx
                        display(HTML(f"<div style='border:1px solid #eee;padding:8px;border-radius:6px;background:#fff'>Best sample index: {idx}<br/>R={best_R:.4g}</div>"))
        best_toggle.observe(_on_best, names='value')

        def _on_save(_):
            fname = RESULTS_DIR / 'feasibility_report_export.json'
            try:
                with open(fname,'w',encoding='utf-8') as fh:
                    _json.dump(report, fh, indent=2)
                save_msg.value = f"<span style='color:green'>Saved: {fname.name}</span>"
            except Exception as e:
                save_msg.value = f"<span style='color:red'>Save failed: {e}</span>"
        save_btn.on_click(_on_save)

        exec_summary = [
            f"Target <b>{R_target:.3g}</b> Hz; fraction ≥ target <b>{frac_above:.1%}</b>; verdict <b>{verdict}</b>.",
            f"Median R = {median_R:.3g}; mean R = {mean_R:.3g}; (P10,P90)=({p10:.3g},{p90:.3g}); best R = {best_R:.3g}."
        ]
        if best_R < R_target:
            exec_summary.append("Best sample below target — improve high-influence parameters (efficiencies, losses, multiplexing).")
        else:
            exec_summary.append("At least one configuration reaches target — explore neighborhood for robustness.")
        exec_html = "<div style='margin-top:10px;padding:8px;border-left:4px solid #888;background:#f9f9f9'>" + " ".join(exec_summary) + "</div>"

        controls = widgets.HBox([best_toggle, save_btn, save_msg])
        left_col = widgets.VBox([widgets.HTML(header_html), widgets.HTML(metrics_html), controls, best_out, widgets.HTML(exec_html), sens_out])
        det_panel = widgets.HBox([left_col, plot_out])
        display(det_panel)
        _register(best_toggle, save_btn, save_msg, plot_out, left_col, det_panel, sens_out, best_out)

    # UI container with R_target control
    r_target_det = widgets.FloatText(value=1e-6, description='R target (detail)')
    render_btn = widgets.Button(description='Render detailed', button_style='info')
    out_det = widgets.Output()

    def _on_render(_):
        out_det.clear_output()
        with out_det:
            display_detailed_report(None, R_target=r_target_det.value)
    render_btn.on_click(_on_render)

    det_container = widgets.VBox([
        widgets.HTML('<b>Detailed feasibility report</b>'),
        widgets.HBox([r_target_det, render_btn]),
        out_det
    ])
    display(det_container)
    _register(det_container, r_target_det, render_btn, out_det)
except Exception as e:
    print('Detailed report UI unavailable:', e)